# FlowSure Modellering — Edge & Cloud Model

**Vak:** ML Engineering & Ops  
**Inlevering:** Week 12 — Modelleren en model tracking  
**Team:** [Vul jullie namen in]  

Dit notebook bouwt twee modellen voor het FlowSure klantenondersteuningssysteem:

- **Edge-model** (Deel A) — TF-IDF + Logistic Regression, geconverteerd naar ONNX voor lichtgewicht inferentie op edge-apparaten
- **Cloud-model** (Deel B) — RAG-systeem met sentence-transformers, FAISS en de Claude API voor antwoord generatie

Beide modellen worden gelogd in MLflow voor experiment tracking en reproduceerbaarheid.

---

# DEEL A — Edge Model: Intent Classificatie

Het edge-model classificeert klantttickets naar intent (27 klassen),
en leidt daar automatisch de categorie en prioriteit uit af.
We vergelijken twee aanpakken: TF-IDF + Logistic Regression (baseline) en DistilBERT (deep learning),
en kiezen het beste model voor edge-deployment.

## A1. Data voorbereiden

We laden de Gold classificatie-tabel, bouwen de intent → categorie/prioriteit mapping,
en maken een train/test split.

In [ ]:
from pyspark.sql.functions import col

# Laad Gold classificatie-data
df = spark.table("flowsure.gold.classification_features")
print(f"Totaal rijen: {df.count():,}")
print(f"Kolommen: {df.columns}")

# Intent → Categorie/Prioriteit mapping (voor lookup na classificatie)
intent_mapping_df = df.select("intent", "category", "priority").distinct().orderBy("category", "intent")
intent_to_cat_prio = {
    row["intent"]: (row["category"], row["priority"])
    for row in intent_mapping_df.collect()
}
print(f"\nUnieke intents: {len(intent_to_cat_prio)}")
intent_mapping_df.show(30, truncate=False)

# Train/test split (80/20)
df_model = df.select("text", "intent")
df_train, df_test = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"\nTraining: {df_train.count():,} rijen")
print(f"Test:     {df_test.count():,} rijen")

# Converteer naar Pandas voor scikit-learn
pdf_train = df_train.toPandas()
pdf_test = df_test.toPandas()
X_train, y_train = pdf_train["text"], pdf_train["intent"]
X_test, y_test = pdf_test["text"], pdf_test["intent"]

## A2. Baseline — TF-IDF + Logistic Regression

We beginnen met het simpelste model als referentiepunt.
TF-IDF zet tekst om naar woordfrequentie-vectoren, en Logistic Regression classificeert daarop.
We gebruiken scikit-learn's Pipeline zodat beide stappen samen als één model werken.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import time

# Bouw en train de pipeline
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

start = time.time()
baseline_model.fit(X_train, y_train)
train_time = time.time() - start

# Evaluatie
start = time.time()
y_pred = baseline_model.predict(X_test)
inference_time = (time.time() - start) / len(X_test) * 1000

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy:     {accuracy:.4f}")
print(f"Train tijd:   {train_time:.1f} seconden")
print(f"Inference:    {inference_time:.3f} ms per ticket")
print(f"\n{classification_report(y_test, y_pred)}")

**Resultaat baseline:** 99.2% accuracy met F1 ≥ 0.96 op alle intents.
Dit hoge resultaat komt doordat de Bitext-dataset synthetisch gegenereerd is:
elke intent heeft duidelijke, onderscheidende formulaties.
In de echte wereld zal de accuracy lager liggen —
maar als baseline voor vergelijking is dit uitstekend.

### A2.1 Baseline loggen in MLflow

We loggen het model, de metrics en de parameters in MLflow.
Zo kunnen we later vergelijken met DistilBERT en het experiment reproduceren.

In [ ]:
import mlflow
import mlflow.sklearn
import joblib
import os

mlflow.set_experiment("/Users/haben1414@gmail.com/flowsure_edge_model_intent")

with mlflow.start_run(run_name="baseline_tfidf_logreg"):
    # Parameters
    mlflow.log_param("model_type", "TF-IDF + Logistic Regression")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("ngram_range", "(1, 2)")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    mlflow.log_param("n_intents", y_train.nunique())

    # Metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("train_time_sec", train_time)
    mlflow.log_metric("inference_ms_per_ticket", inference_time)

    # Model + grootte
    mlflow.sklearn.log_model(baseline_model, "model")
    joblib.dump(baseline_model, "/tmp/baseline_model.joblib")
    model_size_mb = os.path.getsize("/tmp/baseline_model.joblib") / (1024 * 1024)
    mlflow.log_metric("model_size_mb", model_size_mb)

    print(f"MLflow run gelogd!")
    print(f"  Accuracy:       {accuracy:.4f}")
    print(f"  Train tijd:     {train_time:.1f}s")
    print(f"  Inference:      {inference_time:.3f} ms/ticket")
    print(f"  Model grootte:  {model_size_mb:.1f} MB")

## A3. Experiment — DistilBERT Fine-tuning

DistilBERT is een ingekrompen versie van BERT: 40% kleiner, 60% sneller,
maar behoudt 97% van de prestatie. Het begrijpt context en synoniemen,
waar TF-IDF alleen naar losse woorden kijkt.

We fine-tunen het op onze intent-data om te vergelijken met de baseline.

In [ ]:
dbutils.library.restartPython()

In [ ]:
%pip install transformers datasets evaluate accelerate -q

### A3.1 Data klaarmaken voor DistilBERT

DistilBERT verwacht numerieke labels (0, 1, 2, ...) in plaats van tekst.
We maken een mapping en zetten de data om naar het Hugging Face Dataset format.

In [ ]:
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder

# Herlaad data na Python restart
pdf_train = spark.table("flowsure.gold.classification_features").select("text", "intent").toPandas()
from sklearn.model_selection import train_test_split
pdf_train, pdf_test = train_test_split(pdf_train, test_size=0.2, random_state=42, stratify=pdf_train["intent"])

y_train = pdf_train["intent"]

# Labels omzetten naar nummers
le = LabelEncoder()
le.fit(y_train)
pdf_train["label"] = le.transform(pdf_train["intent"])
pdf_test["label"] = le.transform(pdf_test["intent"])

print(f"Aantal klassen: {len(le.classes_)}")
for i in range(5):
    print(f"  {i} → {le.classes_[i]}")

# Converteer naar Hugging Face Datasets
train_ds = Dataset.from_pandas(pdf_train[["text", "label"]])
test_ds = Dataset.from_pandas(pdf_test[["text", "label"]])
print(f"\nTrain: {train_ds}")
print(f"Test:  {test_ds}")

### A3.2 Tokenizen en fine-tunen

DistilBERT leest geen ruwe tekst maar tokens: woorden omgezet naar getallen.
Na tokenisatie fine-tunen we met de Hugging Face Trainer.

In [ ]:
from transformers import AutoTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import os

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_ds_tok = train_ds.map(tokenize, batched=True, batch_size=256)
test_ds_tok = test_ds.map(tokenize, batched=True, batch_size=256)
train_ds_tok.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds_tok.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# DistilBERT model + Trainer
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(le.classes_),
)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metric.compute(predictions=np.argmax(logits, axis=-1), references=labels)

os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "29500"

training_args = TrainingArguments(
    output_dir="/tmp/distilbert_flowsure",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
    use_cpu=True,
    ddp_backend=None,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds_tok, eval_dataset=test_ds_tok,
    compute_metrics=compute_metrics,
)

print("Start fine-tuning...")
trainer.train()
print("Fine-tuning klaar!")

**Opmerking:** DistilBERT fine-tuning op CPU-only Databricks free-tier duurt zeer lang (~4+ uur voor 3 epochs).
Op ons cluster is de training niet volledig afgerond. Dit bevestigt de keuze voor het TF-IDF + LR model:
het traint in seconden, behaalt 99.2% accuracy, en is direct converteerbaar naar ONNX.

### Modelkeuze: waarom TF-IDF + LR als edge-model?

| Criterium | TF-IDF + LR | DistilBERT |
|---|---|---|
| **Accuracy** | 99.2% | ~99% (verwacht) |
| **Model grootte** | 1.53 MB (ONNX) | ~260 MB |
| **Inference snelheid** | 0.14 ms (ONNX) | ~50 ms |
| **CPU compatibel** | ✓ (native) | ✓ (maar traag) |
| **Train tijd** | ~6 seconden | ~4+ uur (CPU) |
| **Edge geschikt** | ✓ (klein, snel) | ✗ (te groot, te traag) |

Het TF-IDF + LR model is de duidelijke winnaar voor edge-deployment:
170x kleiner, 350x sneller, en vergelijkbare accuracy.

## A4. ONNX Conversie — Edge-ready model

We converteren het TF-IDF + LR model naar ONNX formaat voor edge-deployment.
ONNX Runtime is beschikbaar op vrijwel elk platform (mobiel, embedded, server)
en biedt snellere inferentie dan scikit-learn.

In [ ]:
%pip install skl2onnx onnxruntime -q

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import StringTensorType
import onnxruntime as ort
import numpy as np
import time
import os
import json


def convert_to_onnx(sklearn_model, output_path="/tmp/baseline_model.onnx"):
    """Converteer een sklearn pipeline naar ONNX formaat."""
    onnx_model = convert_sklearn(
        sklearn_model, "baseline_tfidf_logreg",
        initial_types=[("text", StringTensorType([None, 1]))],
        target_opset=12,
    )
    with open(output_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    return output_path


def benchmark_onnx(onnx_path, sklearn_model, n_runs=100):
    """Vergelijk ONNX vs sklearn inference snelheid."""
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    single_input = np.array(["I want to cancel my order"]).reshape(-1, 1)

    # ONNX timing
    start = time.time()
    for _ in range(n_runs):
        session.run(None, {input_name: single_input})
    onnx_ms = (time.time() - start) / n_runs * 1000

    # sklearn timing
    start = time.time()
    for _ in range(n_runs):
        sklearn_model.predict(["I want to cancel my order"])
    sklearn_ms = (time.time() - start) / n_runs * 1000

    return session, onnx_ms, sklearn_ms


def classify_ticket_onnx(text, onnx_session, intent_lookup):
    """Classificatie via ONNX runtime: tekst → intent → categorie + prioriteit."""
    input_name = onnx_session.get_inputs()[0].name
    onnx_input = np.array([text]).reshape(-1, 1)
    predicted_intent = onnx_session.run(None, {input_name: onnx_input})[0][0]
    category, priority = intent_lookup.get(predicted_intent, ("ONBEKEND", "unknown"))
    return {"text": text, "intent": predicted_intent, "category": category, "priority": priority}


# 1. Converteer naar ONNX
onnx_path = convert_to_onnx(baseline_model)
onnx_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX model: {onnx_size_mb:.2f} MB")

# 2. Vergelijk output: sklearn vs ONNX
session, onnx_ms, sklearn_ms = benchmark_onnx(onnx_path, baseline_model)

test_tickets = [
    "I want to cancel my order {{Order Number}}",
    "where is my package {{Order Number}}",
    "I need a refund for my purchase",
    "how do I change my password",
    "I have a complaint about your service",
]

input_name = session.get_inputs()[0].name
onnx_input = np.array(test_tickets).reshape(-1, 1)
onnx_preds = session.run(None, {input_name: onnx_input})[0]
sklearn_preds = baseline_model.predict(test_tickets)

print(f"\n{'=' * 60}")
print("VERGELIJKING: sklearn vs ONNX")
print("=" * 60)
for text, sk, ox in zip(test_tickets, sklearn_preds, onnx_preds):
    match = "✓" if sk == ox else "✗"
    print(f"  {match} sklearn: {sk:<25} ONNX: {ox}")

print(f"\n{'=' * 60}")
print(f"INFERENCE SNELHEID (100 runs)")
print("=" * 60)
print(f"  sklearn:  {sklearn_ms:.3f} ms per ticket")
print(f"  ONNX:     {onnx_ms:.3f} ms per ticket")
print(f"  Speedup:  {sklearn_ms / onnx_ms:.1f}x")

# 3. Demo: volledige ONNX flow
print(f"\n{'=' * 60}")
print("DEMO: ONNX — Tekst → Intent → Categorie + Prioriteit")
print("=" * 60)
for ticket in test_tickets:
    result = classify_ticket_onnx(ticket, session, intent_to_cat_prio)
    print(f"\n  Tekst:      {result['text']}")
    print(f"  Intent:     {result['intent']}")
    print(f"  Categorie:  {result['category']}")
    print(f"  Prioriteit: {result['priority']}")

### A4.1 ONNX model + mapping opslaan

In [ ]:
import shutil

volume_path = "/Volumes/flowsure/default/raw_files/models"
dbutils.fs.mkdirs(volume_path)

# ONNX model opslaan
shutil.copy("/tmp/baseline_model.onnx", f"{volume_path}/baseline_model.onnx")

# Intent mapping als JSON
with open(f"{volume_path}/intent_mapping.json", "w") as f:
    json.dump(intent_to_cat_prio, f, indent=2)

# Verificatie
for f in dbutils.fs.ls(volume_path):
    print(f"  {f.name:<30} {f.size / 1024:.1f} KB")
print(f"\nOpgeslagen in: {volume_path}")

---

# DEEL B — Cloud Model: RAG Antwoord Generatie

Het cloud-model genereert antwoordvoorstellen voor klantttickets met een RAG-systeem:

**Waarom RAG in plaats van fine-tuning?**
1. **Geen GPU nodig** — we draaien op Databricks free-tier (CPU only)
2. **Kennisbank updatable** — nieuwe antwoorden toevoegen vereist geen hertraining
3. **Professioneel patroon** — RAG is de standaard aanpak in productie voor klantenservice

**Architectuur:** Klantvraag → embedding → FAISS zoekt vergelijkbare vragen → LLM genereert antwoord

## B1. Setup en data laden

In [ ]:
dbutils.library.restartPython()

In [ ]:
%pip install sentence-transformers faiss-cpu anthropic -q

In [ ]:
from pyspark.sql.functions import col
import numpy as np
import time

# Laad conversatieparen en neem een subsample van 50.000
df_conversations = spark.table("flowsure.gold.conversation_pairs")
total_rows = df_conversations.count()
print(f"Totaal conversatieparen: {total_rows:,}")

SAMPLE_SIZE = 50_000
df_sample = df_conversations.sample(
    fraction=min(SAMPLE_SIZE / total_rows * 1.1, 1.0), seed=42
).limit(SAMPLE_SIZE)

pdf_sample = df_sample.select("customer_text", "support_text", "company").toPandas()
print(f"Subsample: {len(pdf_sample):,} rijen, {pdf_sample['company'].nunique()} bedrijven")
print(f"\n--- Top 5 bedrijven ---")
print(pdf_sample["company"].value_counts().head(5).to_string())

## B2. Embeddings berekenen

We gebruiken `all-MiniLM-L6-v2` (80MB, 384-dimensionale vectors) om elke klantvraag
om te zetten naar een embedding. Vergelijkbare teksten krijgen vergelijkbare vectoren.

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model: all-MiniLM-L6-v2, dimensie: {embed_model.get_embedding_dimension()}")

print(f"\nEmbeddings berekenen voor {len(pdf_sample):,} teksten...")
start = time.time()
embeddings = embed_model.encode(pdf_sample["customer_text"].tolist(), show_progress_bar=False, batch_size=256)
embed_time = time.time() - start

print(f"Klaar in {embed_time:.1f}s ({len(pdf_sample) / embed_time:.0f} teksten/sec)")
print(f"Shape: {embeddings.shape}, geheugen: {embeddings.nbytes / (1024 * 1024):.1f} MB")

## B3. FAISS Index bouwen

FAISS (Facebook AI Similarity Search) vindt de meest vergelijkbare vectoren
in milliseconden over miljoenen vectoren. We gebruiken IndexFlatL2 (exacte search).

In [ ]:
import faiss

# Normaliseer en bouw index
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
dimension = embeddings_normalized.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings_normalized.astype("float32"))

print(f"FAISS index: {index.ntotal:,} vectoren, {dimension}D, ~{index.ntotal * dimension * 4 / (1024**2):.1f} MB")

# Test query
test_query = "I can't pay my bill"
query_emb = embed_model.encode([test_query])
query_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)

start = time.time()
distances, indices = index.search(query_norm.astype("float32"), k=3)
search_ms = (time.time() - start) * 1000

print(f"\nTest: '{test_query}' — {search_ms:.2f} ms")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    print(f"  #{rank} (afstand: {dist:.4f}) {pdf_sample['customer_text'].iloc[idx][:80]}")

## B4. Retrieval functie

In [ ]:
def retrieve_similar(query_text, top_k=3):
    """Zoek de meest vergelijkbare klantvragen in de FAISS index."""
    query_emb = embed_model.encode([query_text])
    query_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    distances, indices = index.search(query_norm.astype("float32"), k=top_k)
    return [
        {"customer_text": pdf_sample["customer_text"].iloc[idx],
         "support_text": pdf_sample["support_text"].iloc[idx],
         "distance": float(dist)}
        for dist, idx in zip(distances[0], indices[0])
    ]


# Test
for query in ["I want to cancel my order 12345", "Your service is terrible"]:
    print(f"\nQUERY: '{query}'")
    for i, r in enumerate(retrieve_similar(query), 1):
        print(f"  #{i} ({r['distance']:.4f}) {r['customer_text'][:70]}")

## B5. LLM antwoord genereren

We combineren retrieval + Claude API: de opgehaalde antwoorden worden als context
meegegeven aan het LLM, dat een nieuw antwoord schrijft.
Als de API niet beschikbaar is, vallen we terug op het beste opgehaalde antwoord.

In [ ]:
import anthropic

try:
    api_key = dbutils.secrets.get(scope="flowsure", key="anthropic-api-key")
    client = anthropic.Anthropic(api_key=api_key)
    LLM_AVAILABLE = True
    print("Anthropic API verbonden")
except Exception:
    LLM_AVAILABLE = False
    print("Geen API key gevonden — fallback naar template-based antwoorden")


PROMPT_TEMPLATE = """Je bent een klantenservice-medewerker bij FlowSure.
Een klant stuurde het volgende bericht:
{customer_text}

Hier zijn vergelijkbare eerdere vragen en hun antwoorden:
{retrieved_context}

Schrijf een professioneel, behulpzaam antwoord voor deze klant.
Houd het kort (2-3 zinnen)."""


def generate_response(customer_text, top_k=3):
    """Volledige RAG pipeline: retrieval + generatie."""
    start = time.time()
    matches = retrieve_similar(customer_text, top_k=top_k)
    retrieval_time = (time.time() - start) * 1000

    context = "\n".join(
        f"Vraag: {m['customer_text']}\nAntwoord: {m['support_text']}" for m in matches
    )

    start = time.time()
    if LLM_AVAILABLE:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=200,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(
                customer_text=customer_text, retrieved_context=context)}],
        )
        answer = response.content[0].text
    else:
        answer = f"[Template] {matches[0]['support_text']}"
    generation_time = (time.time() - start) * 1000

    return {"answer": answer, "retrieval_time_ms": retrieval_time,
            "generation_time_ms": generation_time, "total_time_ms": retrieval_time + generation_time,
            "matches": matches}


# Test met 3 voorbeelden
for ticket in ["I want to cancel my order 12345", "How do I change my shipping address?",
               "Your service is terrible, I want my money back"]:
    result = generate_response(ticket)
    print(f"\nKLANT: '{ticket}'")
    print(f"ANTWOORD: {result['answer'][:120]}")
    print(f"  Retrieval: {result['retrieval_time_ms']:.1f}ms | "
          f"Generatie: {result['generation_time_ms']:.1f}ms | "
          f"Totaal: {result['total_time_ms']:.1f}ms")

## B6. Evaluatie

We meten de kwaliteit en snelheid van het RAG-systeem op 5 test-tickets:
retrieval latency, cosine similarity, en end-to-end latency.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

test_sample = pdf_sample.sample(n=5, random_state=42)
retrieval_latencies, e2e_latencies, similarities = [], [], []

for _, row in test_sample.iterrows():
    query = row["customer_text"]

    # Retrieval + similarity
    start = time.time()
    query_emb = embed_model.encode([query])
    query_norm = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True)
    distances, idxs = index.search(query_norm.astype("float32"), k=3)
    retrieval_latencies.append((time.time() - start) * 1000)

    best_emb = embeddings_normalized[idxs[0][0]].reshape(1, -1)
    similarities.append(cosine_similarity(query_norm, best_emb)[0][0])

    if LLM_AVAILABLE:
        result = generate_response(query)
        e2e_latencies.append(result["total_time_ms"])

r = np.array(retrieval_latencies)
s = np.array(similarities)

print(f"{'=' * 50}")
print(f"EVALUATIE RESULTATEN (5 tickets)")
print(f"{'=' * 50}")
print(f"\nRetrieval latency: gem {r.mean():.1f}ms, mediaan {np.median(r):.1f}ms")
print(f"Cosine similarity: gem {s.mean():.4f}, min {s.min():.4f}")
if e2e_latencies:
    e = np.array(e2e_latencies)
    print(f"End-to-end latency: gem {e.mean():.0f}ms, mediaan {np.median(e):.0f}ms")

## B7. MLflow logging

In [ ]:
import mlflow

mlflow.set_experiment("/Users/haben1414@gmail.com/flowsure_cloud_model")

with mlflow.start_run(run_name="rag_faiss_minilm"):
    mlflow.log_param("model_type", "RAG")
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("embedding_dim", 384)
    mlflow.log_param("index_type", "FAISS_FlatL2")
    mlflow.log_param("top_k", 3)
    mlflow.log_param("subsample_size", len(pdf_sample))
    mlflow.log_param("llm_model", "claude-sonnet-4-20250514" if LLM_AVAILABLE else "fallback_template")

    mlflow.log_metric("avg_retrieval_latency_ms", float(r.mean()))
    mlflow.log_metric("median_retrieval_latency_ms", float(np.median(r)))
    mlflow.log_metric("avg_cosine_similarity", float(s.mean()))
    mlflow.log_metric("min_cosine_similarity", float(s.min()))
    if e2e_latencies:
        mlflow.log_metric("avg_e2e_latency_ms", float(np.array(e2e_latencies).mean()))

    print(f"MLflow run gelogd!")
    print(f"  Retrieval: {r.mean():.2f} ms")
    print(f"  Similarity: {s.mean():.4f}")

### B7.1 RAG artefacten opslaan

In [ ]:
import faiss as faiss_lib

# Clear attrs to avoid serialization error
pdf_sample.attrs.clear()

save_path = "/Workspace/Users/haben1414@gmail.com/flowsure-mlops-support/notebooks"
pdf_sample.to_parquet(f"{save_path}/texts.parquet", index=False)
faiss_lib.write_index(index, f"{save_path}/faiss_index.bin")

print(f"Opgeslagen:")
print(f"  {save_path}/texts.parquet")
print(f"  {save_path}/faiss_index.bin")

---

# Conclusie — Modellering

## Samenvatting

| | Edge-model | Cloud-model |
|---|---|---|
| **Type** | TF-IDF + Logistic Regression (ONNX) | RAG (FAISS + Claude API) |
| **Taak** | Intent classificatie (27 klassen) | Antwoord generatie |
| **Accuracy/kwaliteit** | 99.2% accuracy | Cosine sim. 1.0 (retrieval) |
| **Inference snelheid** | 0.14 ms (ONNX) | ~2.9s (retrieval + LLM) |
| **Model grootte** | 1.53 MB | 80 MB embedding + 73 MB index |
| **Platform** | Edge / mobiel / embedded | Cloud (API vereist) |
| **MLflow experiment** | ✓ gelogd | ✓ gelogd |

## Leerdoelen gedekt

- **Leerdoel 3** — Twee modellen getraind, geëvalueerd, en gelogd in MLflow met relevante metrics
- **Leerdoel 3** — Vergelijking TF-IDF vs DistilBERT met onderbouwde modelkeuze
- **Leerdoel 4** — ONNX conversie voor edge-deployment, intent mapping als JSON artefact